In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROC_DIR = Path("data/processed_company_dataset")
files = sorted(PROC_DIR.glob("*.csv"))
assert len(files) > 0, f"No csv files found in {PROC_DIR.resolve()}"

dfs = []
for fp in files:
    df = pd.read_csv(fp)
    df["date"] = pd.to_datetime(df["date"])
    dfs.append(df)

panel = pd.concat(dfs, ignore_index=True).sort_values(["symbol","date"]).reset_index(drop=True)

print(panel.shape)
panel.head()

(12780, 13)


,date,symbol,log_ret,volatility_20d,volume_change,ps_ratio,pe_ratio,rev_growth_qoq,real_rate,yield_curve,unemployment_change,close,is_trading_day
0,2021-07-03,AAPL,0.000000,0.008942,-1.000000,1.562333e-09,5.922979e-09,0.0,-7.9228,1.68,0.0,139.96,1.0
1,2021-07-04,AAPL,0.000000,0.007711,-1.000000,1.562333e-09,5.922979e-09,0.0,-7.9228,1.68,0.0,139.96,1.0
2,2021-07-05,AAPL,0.000000,0.007400,-1.000000,1.562333e-09,5.922979e-09,0.0,-7.9228,1.68,0.0,139.96,1.0
3,2021-07-06,AAPL,0.014611,0.007783,1.186876,1.585328e-09,6.010157e-09,0.0,-7.9228,1.68,0.0,142.02,1.0
4,2021-07-07,AAPL,0.017796,0.008155,1.103358,1.613793e-09,6.118070e-09,0.0,-7.9228,1.68,0.0,144.57,1.0


In [2]:
# 基础列检查（若缺某些列我会自动跳过）
BASE_COLS = ["date","symbol","log_ret","volatility_20d","volume_change","ps_ratio","pe_ratio",
             "rev_growth_qoq","real_rate","yield_curve","unemployment_change","close","is_trading_day"]

missing = [c for c in BASE_COLS if c not in panel.columns]
print("Missing cols:", missing)

# 仅保留存在的列
use_cols = [c for c in BASE_COLS if c in panel.columns]
panel = panel[use_cols].copy()

# 只用交易日（如果 is_trading_day 有意义）
if "is_trading_day" in panel.columns:
    panel = panel[panel["is_trading_day"] == 1].copy()

# 生成 month key
panel["month"] = panel["date"].dt.to_period("M").astype(str)

panel.head()

Missing cols: []


,date,symbol,log_ret,volatility_20d,volume_change,ps_ratio,pe_ratio,rev_growth_qoq,real_rate,yield_curve,unemployment_change,close,is_trading_day,month
0,2021-07-03,AAPL,0.000000,0.008942,-1.000000,1.562333e-09,5.922979e-09,0.0,-7.9228,1.68,0.0,139.96,1.0,2021-07
1,2021-07-04,AAPL,0.000000,0.007711,-1.000000,1.562333e-09,5.922979e-09,0.0,-7.9228,1.68,0.0,139.96,1.0,2021-07
2,2021-07-05,AAPL,0.000000,0.007400,-1.000000,1.562333e-09,5.922979e-09,0.0,-7.9228,1.68,0.0,139.96,1.0,2021-07
3,2021-07-06,AAPL,0.014611,0.007783,1.186876,1.585328e-09,6.010157e-09,0.0,-7.9228,1.68,0.0,142.02,1.0,2021-07
4,2021-07-07,AAPL,0.017796,0.008155,1.103358,1.613793e-09,6.118070e-09,0.0,-7.9228,1.68,0.0,144.57,1.0,2021-07


In [3]:
def max_drawdown_from_nav(nav: np.ndarray) -> float:
    if len(nav) == 0:
        return np.nan
    peak = np.maximum.accumulate(nav)
    dd = nav / peak - 1.0
    return float(np.min(dd))

def trend_slope(y: np.ndarray) -> float:
    # y: cumulative nav series
    if len(y) < 3:
        return np.nan
    x = np.arange(len(y), dtype=float)
    # 简单线性回归斜率
    x = (x - x.mean()) / (x.std() + 1e-12)
    y = (y - y.mean()) / (y.std() + 1e-12)
    return float((x*y).mean())

rows = []

# 用 close 也可以算月收益（若存在）
has_close = "close" in panel.columns

for (sym, mon), g in panel.groupby(["symbol","month"], sort=True):
    g = g.sort_values("date")
    logret = g["log_ret"].astype(float).to_numpy()
    # 当月净值曲线（从1开始）
    nav = np.exp(np.cumsum(np.nan_to_num(logret, nan=0.0)))
    nav = np.insert(nav, 0, 1.0)  # 加上起点，便于画图更直观
    
    # 当月 summary 特征（pattern）
    month_ret = float(np.exp(np.nansum(logret)) - 1.0)   # 当月简单收益
    month_vol = float(np.nanstd(np.expm1(logret), ddof=0) * np.sqrt(252)) if len(logret) > 1 else np.nan
    month_mdd = max_drawdown_from_nav(nav)
    slope = trend_slope(nav[1:])  # 不含起点
    
    # “曲线形状”更细的特征（可选但很有用）
    # 例如：前半月 vs 后半月收益
    mid = max(1, len(logret)//2)
    ret_first = float(np.exp(np.nansum(logret[:mid])) - 1.0)
    ret_second = float(np.exp(np.nansum(logret[mid:])) - 1.0)

    # 月内波动变化：末尾的 volatility_20d vs 月初
    vol20_start = float(g["volatility_20d"].iloc[0]) if "volatility_20d" in g.columns else np.nan
    vol20_end   = float(g["volatility_20d"].iloc[-1]) if "volatility_20d" in g.columns else np.nan
    vol20_chg   = vol20_end - vol20_start if (np.isfinite(vol20_start) and np.isfinite(vol20_end)) else np.nan

    # 宏观/context（通常对所有 symbol 相同，取均值/末值/变化都可以）
    ctx = {}
    for c in ["real_rate","yield_curve","unemployment_change"]:
        if c in g.columns:
            ctx[f"{c}_mean"] = float(np.nanmean(g[c].astype(float)))
            ctx[f"{c}_end"]  = float(g[c].astype(float).iloc[-1])
            ctx[f"{c}_chg"]  = float(g[c].astype(float).iloc[-1] - g[c].astype(float).iloc[0])

    # fundamentals snapshot（若存在）
    fctx = {}
    for c in ["pe_ratio","ps_ratio","rev_growth_qoq","volume_change"]:
        if c in g.columns:
            fctx[f"{c}_end"] = float(g[c].astype(float).iloc[-1])

    rows.append({
        "symbol": sym,
        "month": mon,
        "start_date": g["date"].iloc[0],
        "end_date": g["date"].iloc[-1],
        "n_days": int(len(g)),
        # curves for explanation
        "curve_logret": logret.tolist(),
        "curve_cum": nav.tolist(),
        # core features (pattern vector pieces)
        "feat_month_ret": month_ret,
        "feat_month_vol": month_vol,
        "feat_month_mdd": month_mdd,
        "feat_trend_slope": slope,
        "feat_ret_first_half": ret_first,
        "feat_ret_second_half": ret_second,
        "feat_vol20_chg": vol20_chg,
        # context
        **{f"ctx_{k}": v for k,v in ctx.items()},
        **{f"ctx_{k}": v for k,v in fctx.items()},
    })

monthly_chunks = pd.DataFrame(rows).sort_values(["symbol","month"]).reset_index(drop=True)
print(monthly_chunks.shape)
monthly_chunks.head()

(420, 27)


,symbol,month,start_date,end_date,n_days,curve_logret,curve_cum,feat_month_ret,feat_month_vol,feat_month_mdd,...,ctx_yield_curve_mean,ctx_yield_curve_end,ctx_yield_curve_chg,ctx_unemployment_change_mean,ctx_unemployment_change_end,ctx_unemployment_change_chg,ctx_pe_ratio_end,ctx_ps_ratio_end,ctx_rev_growth_qoq_end,ctx_volume_change_end
0,AAPL,2021-07,2021-07-03,2021-07-31,29,"[0.0, 0.0, 0.0, 0.0146112252544602, 0.01779592...","[1.0, 1.0, 1.0, 1.0, 1.0147184909974278, 1.032...",0.042155,0.178651,-0.044921,...,1.68,1.68,0.0,0.0,0.0,0.0,6.172662e-09,1.628193e-09,0.000000,-1.000000
1,AAPL,2021-08,2021-08-01,2021-08-31,31,"[0.0, -0.0023337233462202, 0.0125650382971298,...","[1.0, 1.0, 0.9976689976689976, 1.0102838338132...",0.040930,0.151949,-0.031498,...,1.68,1.68,0.0,0.0,0.0,0.0,6.425307e-09,1.694834e-09,0.000000,0.708916
2,AAPL,2021-09,2021-09-01,2021-09-30,30,"[0.0044686937739958, 0.0074471209084012, 0.004...","[1.0, 1.0044786932753735, 1.0119870908252648, ...",-0.068037,0.171650,-0.096943,...,1.68,1.68,0.0,0.0,0.0,0.0,6.507542e-09,1.737603e-09,-0.090976,0.443733
3,AAPL,2021-10,2021-10-01,2021-10-31,31,"[0.0080943605762167, 0.0, 0.0, -0.024913457164...","[1.0, 1.0081272084805655, 1.0081272084805655, ...",0.058657,0.152249,-0.024606,...,1.68,1.68,0.0,0.0,0.0,0.0,6.889257e-09,1.839526e-09,-0.090976,-1.000000
4,AAPL,2021-11,2021-11-01,2021-11-30,30,"[-0.0056232575543622, 0.0070908050127334, 0.00...","[1.0, 0.9943925233644859, 1.0014686248331108, ...",0.103471,0.192010,-0.031678,...,1.68,1.68,0.0,0.0,0.0,0.0,7.602097e-09,2.029865e-09,-0.090976,1.926668


In [4]:
monthly_chunks = monthly_chunks.sort_values(["symbol","month"]).reset_index(drop=True)

monthly_chunks["label_next_month_ret"] = monthly_chunks.groupby("symbol")["feat_month_ret"].shift(-1)
monthly_chunks["label_next_month_mdd"] = monthly_chunks.groupby("symbol")["feat_month_mdd"].shift(-1)

# 最后一月没有 next label，会是 NaN（正常）
monthly_chunks.tail()

,symbol,month,start_date,end_date,n_days,curve_logret,curve_cum,feat_month_ret,feat_month_vol,feat_month_mdd,...,ctx_yield_curve_chg,ctx_unemployment_change_mean,ctx_unemployment_change_end,ctx_unemployment_change_chg,ctx_pe_ratio_end,ctx_ps_ratio_end,ctx_rev_growth_qoq_end,ctx_volume_change_end,label_next_month_ret,label_next_month_mdd
415,TXN,2024-08,2024-08-01,2024-08-31,31,"[-0.0527896139538773, -0.0308330840260709, 0.0...","[1.0, 0.948579559393553, 0.9197782248172319, 0...",0.051666,0.330773,-0.110986,...,0.00,0.023643,0.00000,-0.05000,1.939729e-07,5.854685e-08,-0.102036,-1.000000,-0.036251,-0.085658
416,TXN,2024-09,2024-09-01,2024-09-30,30,"[0.0, 0.0, -0.0601377803728232, 0.010694234932...","[1.0, 1.0, 1.0, 0.9416347858542504, 0.95175888...",-0.036251,0.267645,-0.085658,...,0.05,0.000000,0.00000,0.00000,1.832919e-07,5.404762e-08,0.043977,0.468672,-0.016508,-0.070379
417,TXN,2024-10,2024-10-01,2024-10-31,31,"[-0.0243538018129477, 0.0090364291666469, -0.0...","[1.0, 0.9759403592002712, 0.9847993416275356, ...",-0.016508,0.248129,-0.070379,...,0.00,-0.022273,-0.02381,-0.02381,1.802662e-07,5.315542e-08,0.043977,0.608666,-0.010484,-0.101185
418,TXN,2024-11,2024-11-01,2024-11-30,30,"[0.0086258131379573, 0.0, 0.0, -0.006167722327...","[1.0, 1.0086631226619414, 1.0086631226619414, ...",-0.010484,0.280992,-0.101185,...,0.00,-0.000794,0.00000,0.02381,1.783762e-07,5.259812e-08,0.043977,-1.000000,-0.067254,-0.085787
419,TXN,2024-12,2024-12-01,2024-12-31,31,"[0.0, 0.0037238443270441, -0.0228074345648212,...","[1.0, 1.0, 1.0037307864497835, 0.9810973486544...",-0.067254,0.141714,-0.085787,...,0.19,0.023603,0.00000,-0.02439,1.376725e-07,4.517225e-08,0.086081,-0.051848,NaN,NaN


In [5]:
# 你可以随时增删这个列表（这些就是 embedding 的 feature space）
FEATURE_COLS = [
    "feat_month_ret",
    "feat_month_vol",
    "feat_month_mdd",
    "feat_trend_slope",
    "feat_ret_first_half",
    "feat_ret_second_half",
    "feat_vol20_chg",
    # macro context
    "ctx_yield_curve_end",
    "ctx_yield_curve_chg",
    "ctx_real_rate_end",
    "ctx_real_rate_chg",
    "ctx_unemployment_change_end",
    "ctx_unemployment_change_chg",
    # fundamentals snapshot
    "ctx_pe_ratio_end",
    "ctx_ps_ratio_end",
    "ctx_rev_growth_qoq_end",
    "ctx_volume_change_end",
]

# 只用存在的列
FEATURE_COLS = [c for c in FEATURE_COLS if c in monthly_chunks.columns]
print("Using features:", FEATURE_COLS)

X = monthly_chunks[FEATURE_COLS].astype(float).copy()

# 处理缺失：先用每列中位数填充（稳健）
X = X.apply(lambda s: s.fillna(s.median()), axis=0)

# 标准化（用于 kNN 相似度）
mu = X.mean(axis=0)
sd = X.std(axis=0, ddof=0).replace(0, 1.0)
Xz = (X - mu) / sd

# 存回去，方便后续检索
monthly_chunks["feat_vec"] = list(Xz.to_numpy())

monthly_chunks[["symbol","month"] + FEATURE_COLS + ["label_next_month_ret"]].head()

Using features: ['feat_month_ret', 'feat_month_vol', 'feat_month_mdd', 'feat_trend_slope', 'feat_ret_first_half', 'feat_ret_second_half', 'feat_vol20_chg', 'ctx_yield_curve_end', 'ctx_yield_curve_chg', 'ctx_real_rate_end', 'ctx_real_rate_chg', 'ctx_unemployment_change_end', 'ctx_unemployment_change_chg', 'ctx_pe_ratio_end', 'ctx_ps_ratio_end', 'ctx_rev_growth_qoq_end', 'ctx_volume_change_end']


,symbol,month,feat_month_ret,feat_month_vol,feat_month_mdd,feat_trend_slope,feat_ret_first_half,feat_ret_second_half,feat_vol20_chg,ctx_yield_curve_end,ctx_yield_curve_chg,ctx_real_rate_end,ctx_real_rate_chg,ctx_unemployment_change_end,ctx_unemployment_change_chg,ctx_pe_ratio_end,ctx_ps_ratio_end,ctx_rev_growth_qoq_end,ctx_volume_change_end,label_next_month_ret
0,AAPL,2021-07,0.042155,0.178651,-0.044921,0.637531,0.045942,-0.003620,0.003488,1.68,0.0,-7.9228,0.0,0.0,0.0,6.172662e-09,1.628193e-09,0.000000,-1.000000,0.040930
1,AAPL,2021-08,0.040930,0.151949,-0.031498,0.671805,0.022213,0.018310,-0.000766,1.68,0.0,-7.9228,0.0,0.0,0.0,6.425307e-09,1.694834e-09,0.000000,0.708916,-0.068037
2,AAPL,2021-09,-0.068037,0.171650,-0.096943,-0.892885,-0.018442,-0.050527,-0.000579,1.68,0.0,-7.9228,0.0,0.0,0.0,6.507542e-09,1.737603e-09,-0.090976,0.443733,0.058657
3,AAPL,2021-10,0.058657,0.152249,-0.024606,0.906171,0.023604,0.034245,-0.000684,1.68,0.0,-7.9228,0.0,0.0,0.0,6.889257e-09,1.839526e-09,-0.090976,-1.000000,0.103471
4,AAPL,2021-11,0.103471,0.192010,-0.031678,0.816157,0.001335,0.102000,0.003903,1.68,0.0,-7.9228,0.0,0.0,0.0,7.602097e-09,2.029865e-09,-0.090976,1.926668,0.074229


In [6]:
# Parquet 保存 list 列更友好（也可以用 pickle）
out_parquet = "monthly_chunks.parquet"
monthly_chunks.to_parquet(out_parquet, index=False)
print("Saved:", out_parquet)

# 同时保存向量矩阵与标准化参数（用于线上一致性）
np.save("monthly_feat_matrix.npy", Xz.to_numpy())
mu.to_csv("feat_mu.csv")
sd.to_csv("feat_sd.csv")
print("Saved: monthly_feat_matrix.npy, feat_mu.csv, feat_sd.csv")

Saved: monthly_chunks.parquet
Saved: monthly_feat_matrix.npy, feat_mu.csv, feat_sd.csv
